# Sprint 2.4 - Alvo e features

Este notebook cria o alvo `qualidade_inadequada_24h` e as features iniciais usadas nas proximas sprints.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yaml

ROOT = Path.cwd()
if not (ROOT / 'data' / 'interim' / 'integrado_limpo.csv').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'src'))
from prepara_sprint2 import add_features, target_sources, temporal_split

config = yaml.safe_load((ROOT / 'config' / 'params.yaml').read_text(encoding='utf-8'))
sprint2 = config['sprint2']
cleaned = pd.read_csv(ROOT / 'data' / 'interim' / 'integrado_limpo.csv', parse_dates=['timestamp'])
train, test, cutoff = temporal_split(cleaned, sprint2['test_fraction'])

source_flags = target_sources(cleaned, sprint2['target'])
bad_now = source_flags.any(axis=1)
target_name = sprint2['target']['name']
target = bad_now.shift(-sprint2['forecast_horizon_hours']).astype('Int64')

featured = add_features(cleaned)
featured[target_name] = target
feature_columns = [column for column in featured.columns if column not in ('timestamp', target_name)]

train_features = featured.loc[featured['timestamp'] < cutoff].dropna(subset=[target_name])
test_features = featured.loc[featured['timestamp'] >= cutoff].dropna(subset=[target_name])
train_model = train_features.dropna(subset=feature_columns)
test_model = test_features.dropna(subset=feature_columns)

out_dir = ROOT / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
train_model.to_csv(out_dir / 'treino_features.csv', index=False, encoding='utf-8')
test_model.to_csv(out_dir / 'teste_features.csv', index=False, encoding='utf-8')

{
    'target': target_name,
    'horizon_hours': sprint2['forecast_horizon_hours'],
    'feature_columns': len(feature_columns),
    'train_rows': len(train_model),
    'test_rows': len(test_model),
    'train_target_counts': train_model[target_name].value_counts().sort_index().to_dict(),
    'test_target_counts': test_model[target_name].value_counts().sort_index().to_dict(),
}